### The goal of this exercise is to use the dataset of Sleep & Doomscrolling Habits from Kaggle to ask questions and use SQL to answer said questions.

In [1]:
# First we will Import our resources and CSV file to perform our queries on

In [2]:
import duckdb as db

In [3]:
doom = db.read_csv("data/sleep_doomscrolling_habits.csv").df()
type(doom)

pandas.DataFrame

In [4]:
sql = db.sql

In [28]:
#check to see what our table looks like
doom

,respondent_id,age,bedtime_screen_time_minutes,total_daily_screen_time_hours,doomscroll_sessions_per_night,avg_doomscroll_session_minutes,sleep_hours_per_night,sleep_latency_minutes,number_of_night_wakeups,caffeine_intake_mg_per_day,...,occupation_status,country_region,primary_device_used_at_night,bedtime_routine_type,uses_night_mode,keeps_phone_in_bedroom,consumes_negative_news_content,uses_sleep_tracking_app,doomscroller,sleep_quality_category
0,D30000,22,13.0,1.85,2,5.4,8.1,19.0,2,155.0,...,Unemployed,Germany,Tablet,Meditation/Journaling,True,True,True,False,True,Good
1,D30001,18,90.0,3.61,3,29.6,7.0,31.0,3,82.0,...,Student,India,Smartphone,Meditation/Journaling,False,True,True,False,True,Poor
2,D30002,37,51.0,5.15,3,15.5,7.1,24.0,2,49.0,...,Student,India,Laptop,Reading,False,True,False,True,True,Fair
3,D30003,27,53.0,3.74,0,51.4,8.4,35.0,0,42.0,...,NaN,United States,Smartphone,No Fixed Routine,False,True,True,False,False,Fair
4,D30004,17,67.0,6.09,2,32.2,7.2,19.0,1,76.0,...,Employed Part-time,United States,Smartphone,Watching Videos,False,False,False,False,False,Fair
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,D30995,26,85.0,3.56,4,25.4,7.6,34.0,1,169.0,...,Student,India,Tablet,Meditation/Journaling,False,True,True,True,True,Poor
996,D30996,23,3.0,1.10,1,10.7,7.5,31.0,1,34.0,...,Student,United States,Smartphone,Scrolling Social Media,False,True,False,False,False,Fair
997,D30997,15,28.0,4.38,1,30.6,9.2,20.0,0,100.0,...,Employed Full-time,United States,Smartphone,Scrolling Social Media,False,True,False,False,False,Good
998,D30998,22,23.0,1.84,0,27.9,6.9,41.0,2,45.0,...,Student,India,Tablet,Watching Videos,False,True,True,True,False,Fair


### Now we need to ask ourselves, what do we want to know? 
#### The initial question I want to ask is, What is the average amount of sleep per age group?

In [6]:
sql("""
SELECT 
    age,
    ROUND(AVG(sleep_hours_per_night),2) AS average_sleep
FROM doom
GROUP BY age
ORDER BY average_sleep ASC;
""")

┌───────┬───────────────┐
│  age  │ average_sleep │
│ int64 │    double     │
├───────┼───────────────┤
│    54 │           6.5 │
│    42 │          7.09 │
│    37 │          7.12 │
│    35 │          7.14 │
│    28 │          7.17 │
│    30 │          7.23 │
│    34 │          7.24 │
│    40 │          7.26 │
│    24 │          7.28 │
│    43 │          7.29 │
│     · │            ·  │
│     · │            ·  │
│     · │            ·  │
│    36 │          7.58 │
│    31 │          7.58 │
│    39 │          7.62 │
│    50 │          7.65 │
│    29 │          7.68 │
│    32 │          7.69 │
│    41 │           7.7 │
│    19 │          7.72 │
│    44 │          7.88 │
│    49 │           7.9 │
└───────┴───────────────┘
  36 rows     2 columns
  (20 shown)            

In [7]:
#this doesn't look very good and is a little hard to read, lets check if gender plays a part.
sql("""
SELECT gender,
ROUND(AVG(sleep_hours_per_night),2) AS average_sleep
FROM doom
GROUP BY gender
""")

┌───────────────────┬───────────────┐
│      gender       │ average_sleep │
│      varchar      │    double     │
├───────────────────┼───────────────┤
│ Male              │          7.39 │
│ Female            │          7.44 │
│ Prefer not to say │          7.44 │
└───────────────────┴───────────────┘

In [24]:
#Male seems to have slightly worse sleep but not by a lot,lets leave the gender grouping behind and group them by age ranges; since most of our data is between 15 and 38 we'll group everything over 37 and Under 18. 
#I want to break it up into even chunks of 5 years.
sql("""
SELECT
    CASE
        WHEN age < 18 THEN 'Under 18'
        WHEN age BETWEEN 18 AND 22 THEN '18-22'
        WHEN age BETWEEN 23 AND 27 THEN '23-27'
        WHEN age BETWEEN 28 AND 32 THEN '28-32'
        WHEN age BETWEEN 33 AND 37 THEN '33-37'
        ELSE 'Over 37'
    END AS age_range,
    ROUND(AVG(sleep_hours_per_night),2) AS average_sleep
FROM doom
GROUP BY age_range
ORDER BY average_sleep DESC;    
""")

┌───────────┬───────────────┐
│ age_range │ average_sleep │
│  varchar  │    double     │
├───────────┼───────────────┤
│ 18-22     │          7.48 │
│ 28-32     │          7.46 │
│ Over 37   │          7.43 │
│ 23-27     │          7.42 │
│ Under 18  │           7.4 │
│ 33-37     │          7.28 │
└───────────┴───────────────┘

#### We can see that the 18-22 age range has the best average sleep
##### Are there any factors that affect that? Let's check how much caffeine everyone drinks and bedtime screen time to see if they have an impact on the average

In [9]:
sql("""
SELECT
    CASE
        WHEN age < 18 THEN 'Under 18'
        WHEN age BETWEEN 18 AND 22 THEN '18-22'
        WHEN age BETWEEN 23 AND 27 THEN '23-27'
        WHEN age BETWEEN 28 AND 32 THEN '28-32'
        WHEN age BETWEEN 33 AND 37 THEN '33-37'
        ELSE 'Over 37'
    END AS age_range,
    ROUND(AVG(sleep_hours_per_night),2) AS average_sleep,
    ROUND(AVG(bedtime_screen_time_minutes),2) AS average_betime_screen,
    ROUND(AVG(caffeine_intake_mg_per_day),2) AS average_daily_caffeine_mg
FROM doom
GROUP BY age_range
ORDER BY average_sleep DESC;    
""")

┌───────────┬───────────────┬───────────────────────┬───────────────────────────┐
│ age_range │ average_sleep │ average_betime_screen │ average_daily_caffeine_mg │
│  varchar  │    double     │        double         │          double           │
├───────────┼───────────────┼───────────────────────┼───────────────────────────┤
│ 18-22     │          7.48 │                 50.84 │                    115.67 │
│ 28-32     │          7.46 │                 52.19 │                    122.96 │
│ Over 37   │          7.43 │                  49.2 │                    114.62 │
│ 23-27     │          7.42 │                 48.36 │                    117.79 │
│ Under 18  │           7.4 │                 49.31 │                    121.07 │
│ 33-37     │          7.28 │                 50.21 │                    131.22 │
└───────────┴───────────────┴───────────────────────┴───────────────────────────┘

In [10]:
#those didn't seem to have much of an affect on how much sleep each person gets, let's see if waking up in the middle of the night and number of phone checks per night has an effect.
sql("""
SELECT
    CASE
        WHEN age < 18 THEN 'Under 18'
        WHEN age BETWEEN 18 AND 22 THEN '18-22'
        WHEN age BETWEEN 23 AND 27 THEN '23-27'
        WHEN age BETWEEN 28 AND 32 THEN '28-32'
        WHEN age BETWEEN 33 AND 37 THEN '33-37'
        ELSE 'Over 37'
    END AS age_range,
    ROUND(AVG(sleep_hours_per_night),2) AS average_sleep,
    ROUND(AVG(bedtime_screen_time_minutes),2) AS average_betime_screen,
    ROUND(AVG(caffeine_intake_mg_per_day),2) AS average_daily_caffeine_mg,
    ROUND(AVG(number_of_night_wakeups),2) AS average_wakeups,
    ROUND(AVG(phone_checks_per_night),2) AS average_phone_checks
FROM doom
GROUP BY age_range
ORDER BY average_sleep DESC;    
""")


┌───────────┬───────────────┬───────────────────────┬───────────────────────────┬─────────────────┬──────────────────────┐
│ age_range │ average_sleep │ average_betime_screen │ average_daily_caffeine_mg │ average_wakeups │ average_phone_checks │
│  varchar  │    double     │        double         │          double           │     double      │        double        │
├───────────┼───────────────┼───────────────────────┼───────────────────────────┼─────────────────┼──────────────────────┤
│ 18-22     │          7.48 │                 50.84 │                    115.67 │            1.52 │                 2.62 │
│ 28-32     │          7.46 │                 52.19 │                    122.96 │            1.44 │                 2.49 │
│ Over 37   │          7.43 │                  49.2 │                    114.62 │            1.33 │                 2.53 │
│ 23-27     │          7.42 │                 48.36 │                    117.79 │            1.38 │                  2.4 │
│ Under 18  │   

### Let's change how we're looking at this, averaging out the age groups doesnt seem to be showing us data that shows large enough changes.
#### Let's group by how many hours of sleep people are getting and their screen time.

In [11]:
sql("""
SELECT
    CASE
        WHEN sleep_hours_per_night < 6 THEN 'Under 6'
        WHEN sleep_hours_per_night BETWEEN 6 AND 7 THEN '6-7'
        WHEN sleep_hours_per_night BETWEEN 7 AND 8 THEN '7-8'
        WHEN sleep_hours_per_night BETWEEN 8 AND 9 THEN '8-9'
        ELSE 'Over 9'
    END AS sleep_time,
    ROUND(AVG(bedtime_screen_time_minutes),2) AS average_screen
FROM doom,
GROUP BY Sleep_time
order by average_screen DESC;
""")
    

┌────────────┬────────────────┐
│ sleep_time │ average_screen │
│  varchar   │     double     │
├────────────┼────────────────┤
│ Under 6    │         100.87 │
│ 6-7        │          63.21 │
│ 7-8        │          45.99 │
│ 8-9        │          34.69 │
│ Over 9     │          31.31 │
└────────────┴────────────────┘

In [12]:
#We can really see here that screen time before bed can affect how long you sleep, The amount of sleep trends down the more screentime there is before bed. Let's see if anything else affects it that much.
sql("""
SELECT
    CASE
        WHEN sleep_hours_per_night < 6 THEN 'Under 6'
        WHEN sleep_hours_per_night BETWEEN 6 AND 7 THEN '6-7'
        WHEN sleep_hours_per_night BETWEEN 7 AND 8 THEN '7-8'
        WHEN sleep_hours_per_night BETWEEN 8 AND 9 THEN '8-9'
        ELSE 'Over 9'
    END AS sleep_time,
    ROUND(AVG(bedtime_screen_time_minutes),2) AS average_screen,
    ROUND(AVG(exercise_minutes_per_day),2) AS average_exercise,
    ROUND(AVG(caffeine_intake_mg_per_day),2) AS average_caffeine,
    ROUND(AVG(stress_score)) AS average_stress
FROM doom
GROUP BY sleep_time
ORDER BY average_screen DESC
""")
    
    
    

┌────────────┬────────────────┬──────────────────┬──────────────────┬────────────────┐
│ sleep_time │ average_screen │ average_exercise │ average_caffeine │ average_stress │
│  varchar   │     double     │      double      │      double      │     double     │
├────────────┼────────────────┼──────────────────┼──────────────────┼────────────────┤
│ Under 6    │         100.87 │            50.79 │           125.69 │            7.0 │
│ 6-7        │          63.21 │            49.45 │           119.62 │            6.0 │
│ 7-8        │          45.99 │            54.77 │           119.04 │            6.0 │
│ 8-9        │          34.69 │            54.95 │           126.31 │            6.0 │
│ Over 9     │          31.31 │             59.2 │           120.14 │            6.0 │
└────────────┴────────────────┴──────────────────┴──────────────────┴────────────────┘

## Screen time still seems to be most obvious factor, exercise looks like it plays a minor role and stress seems to be higher with lack of sleep. Caffeine looks like it has almost no impact on sleep time.
#### We can see that screen time has a pretty large impact on how many hours someone sleeps, lets see if there is also a change in quality of sleep.


In [13]:

sql("""
SELECT
    CASE
        WHEN sleep_hours_per_night < 6 THEN 'Under 6'
        WHEN sleep_hours_per_night BETWEEN 6 AND 7 THEN '6-7'
        WHEN sleep_hours_per_night BETWEEN 7 AND 8 THEN '7-8'
        WHEN sleep_hours_per_night BETWEEN 8 AND 9 THEN '8-9'
        ELSE 'Over 9'
    END AS sleep_time,
    ROUND(AVG(bedtime_screen_time_minutes),2) AS average_screen,
    ROUND(AVG(exercise_minutes_per_day),2) AS average_exercise,
    ROUND(AVG(caffeine_intake_mg_per_day),2) AS average_caffeine,
    ROUND(AVG(stress_score)) AS average_stress,
    sleep_quality_category
FROM doom
GROUP BY sleep_time, sleep_quality_category
ORDER BY average_screen DESC
""")


┌────────────┬────────────────┬──────────────────┬──────────────────┬────────────────┬────────────────────────┐
│ sleep_time │ average_screen │ average_exercise │ average_caffeine │ average_stress │ sleep_quality_category │
│  varchar   │     double     │      double      │      double      │     double     │        varchar         │
├────────────┼────────────────┼──────────────────┼──────────────────┼────────────────┼────────────────────────┤
│ Under 6    │         106.67 │            53.17 │           118.39 │            7.0 │ Poor                   │
│ Under 6    │         100.33 │             49.0 │            137.0 │            6.0 │ Fair                   │
│ 6-7        │          77.32 │            47.57 │            111.8 │            6.0 │ Poor                   │
│ 7-8        │          58.26 │             53.1 │           132.75 │            6.0 │ Poor                   │
│ Under 6    │           55.0 │            33.67 │           170.33 │            7.0 │ Good             

In [14]:
#It does not look like exercise affects the quality of sleep, it does seem like the lower the screentime the better quality of the sleep though. Maybe the type of screen matters?
sql("""
SELECT
    CASE
        WHEN sleep_hours_per_night < 6 THEN 'Under 6'
        WHEN sleep_hours_per_night BETWEEN 6 AND 7 THEN '6-7'
        WHEN sleep_hours_per_night BETWEEN 7 AND 8 THEN '7-8'
        WHEN sleep_hours_per_night BETWEEN 8 AND 9 THEN '8-9'
        ELSE 'Over 9'
    END AS sleep_time,
    ROUND(AVG(bedtime_screen_time_minutes),2) AS average_screen,
    ROUND(AVG(stress_score)) AS average_stress,
    primary_device_used_at_night,
FROM doom
WHERE primary_device_used_at_night IS NOT NULL
GROUP BY sleep_time, primary_device_used_at_night
ORDER BY average_screen DESC
""").show(max_rows=100)


┌────────────┬────────────────┬────────────────┬──────────────────────────────┐
│ sleep_time │ average_screen │ average_stress │ primary_device_used_at_night │
│  varchar   │     double     │     double     │           varchar            │
├────────────┼────────────────┼────────────────┼──────────────────────────────┤
│ Under 6    │         113.17 │            8.0 │ Laptop                       │
│ Under 6    │         103.29 │            7.0 │ Smartphone                   │
│ 6-7        │          64.92 │            6.0 │ Smartphone                   │
│ 6-7        │          64.87 │            6.0 │ Laptop                       │
│ 6-7        │          61.69 │            6.0 │ TV                           │
│ Under 6    │           61.5 │            5.0 │ TV                           │
│ Under 6    │           55.0 │            6.0 │ Tablet                       │
│ 6-7        │          50.73 │            6.0 │ Tablet                       │
│ 7-8        │          48.36 │         

In [15]:
#it doesn't seem too much like the type of screen matters, just how much screentime. We can see that the Under 6 hours of sleep group has high stress and high screentime with the laptop, I wonder if they are students? Let's find out.
sql("""
SELECT
    CASE
        WHEN sleep_hours_per_night < 6 THEN 'Under 6'
        WHEN sleep_hours_per_night BETWEEN 6 AND 7 THEN '6-7'
        WHEN sleep_hours_per_night BETWEEN 7 AND 8 THEN '7-8'
        WHEN sleep_hours_per_night BETWEEN 8 AND 9 THEN '8-9'
        ELSE 'Over 9'
    END AS sleep_time,
    ROUND(AVG(bedtime_screen_time_minutes),2) AS average_screen,
    ROUND(AVG(stress_score)) AS average_stress,
    primary_device_used_at_night AS device,
    occupation_status
FROM doom
WHERE device = 'Laptop'
GROUP BY sleep_time, primary_device_used_at_night, occupation_status
ORDER BY average_screen DESC
""").show(max_rows=100)


┌────────────┬────────────────┬────────────────┬─────────┬────────────────────┐
│ sleep_time │ average_screen │ average_stress │ device  │ occupation_status  │
│  varchar   │     double     │     double     │ varchar │      varchar       │
├────────────┼────────────────┼────────────────┼─────────┼────────────────────┤
│ Under 6    │          125.2 │            8.0 │ Laptop  │ Employed Full-time │
│ 6-7        │           92.0 │            6.0 │ Laptop  │ Employed Part-time │
│ 6-7        │           77.6 │            6.0 │ Laptop  │ Employed Full-time │
│ 6-7        │          61.67 │            6.0 │ Laptop  │ Student            │
│ 7-8        │          55.88 │            6.0 │ Laptop  │ Employed Full-time │
│ Under 6    │           53.0 │            4.0 │ Laptop  │ Student            │
│ 8-9        │           52.5 │            9.0 │ Laptop  │ NULL               │
│ 6-7        │          52.33 │            5.0 │ Laptop  │ NULL               │
│ 8-9        │           51.0 │         

In [16]:
#Doesn't seem like the occupation or device type really matters, lets change gears. Maybe bedtime routine will have a specific affect on sleep.
sql("""
SELECT
    ROUND(AVG(sleep_hours_per_night),2) AS average_sleep,
    bedtime_routine_type
FROM doom
WHERE bedtime_routine_type IS NOT NULL
GROUP BY bedtime_routine_type
ORDER BY average_sleep DESC;
""").show(max_rows=100)

┌───────────────┬────────────────────────┐
│ average_sleep │  bedtime_routine_type  │
│    double     │        varchar         │
├───────────────┼────────────────────────┤
│          7.46 │ Watching Videos        │
│          7.44 │ Meditation/Journaling  │
│          7.43 │ No Fixed Routine       │
│          7.39 │ Scrolling Social Media │
│          7.36 │ Reading                │
└───────────────┴────────────────────────┘



In [17]:
## A little surprising that reading has the lowest average sleep and watching videos has the highest. 

In [29]:
sql("""
SELECT
    CASE
        WHEN sleep_hours_per_night < 6 THEN 'Under 6'
        WHEN sleep_hours_per_night BETWEEN 6 AND 7 THEN '6-7'
        WHEN sleep_hours_per_night BETWEEN 7 AND 8 THEN '7-8'
        WHEN sleep_hours_per_night BETWEEN 8 AND 9 THEN '8-9'
        ELSE 'Over 9'
    END AS sleep_time,
    COUNT(*)
FROM doom
GROUP BY sleep_time
""").show(max_rows=100)

┌────────────┬──────────────┐
│ sleep_time │ count_star() │
│  varchar   │    int64     │
├────────────┼──────────────┤
│ 6-7        │          278 │
│ 7-8        │          492 │
│ Under 6    │           30 │
│ 8-9        │          184 │
│ Over 9     │           16 │
└────────────┴──────────────┘



In [30]:
sql("""
SELECT
    CASE
        WHEN age < 18 THEN 'Under 18'
        WHEN age BETWEEN 18 AND 22 THEN '18-22'
        WHEN age BETWEEN 23 AND 27 THEN '23-27'
        WHEN age BETWEEN 28 AND 32 THEN '28-32'
        WHEN age BETWEEN 33 AND 37 THEN '33-37'
        ELSE 'Over 37'
    END AS age_range,
    Count(*)
FROM doom
GROUP BY age_range    
""")

┌───────────┬──────────────┐
│ age_range │ count_star() │
│  varchar  │    int64     │
├───────────┼──────────────┤
│ 33-37     │          157 │
│ 23-27     │          244 │
│ Under 18  │          108 │
│ Over 37   │           94 │
│ 28-32     │          217 │
│ 18-22     │          180 │
└───────────┴──────────────┘